In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [22]:
BUSINESS_SCALE = 3.5
COST_SCALE = 5.0
INEFFICIENCY_MULTIPLIER = 1.35
TARGET_ROWS = 5000
SEED = 42

In [23]:
# Reproducibility
SEED = 42
np.random.seed(SEED)

# Where to write outputs
OUTPUT_DIR = Path(".")  # notebook folder
BILLING_PATH = OUTPUT_DIR / "billing.csv"
DICTIONARY_PATH = OUTPUT_DIR / "data_dictionary.csv"

# Time range & target size
START_DATE = "2025-10-01"
END_DATE = "2026-01-28"
TARGET_ROWS = 5000

dates = pd.date_range(START_DATE, END_DATE, freq="D")


# How much bigger the business is (customers, workflows, revenue)
BUSINESS_SCALE = 3.5

# Overall cost scale (cloud infra spend)
COST_SCALE = 5.0

# Extra inefficiency after acquisition (applied to selected anomaly cases)
INEFFICIENCY_MULTIPLIER = 1.35

In [24]:
# Accounts, products, services, tags

accounts = [
    {"account_id": "1001", "account_name": "legacy-core-prod", "business_unit": "core-platform"},
    {"account_id": "1002", "account_name": "acquired-ai-platform", "business_unit": "ai-products"},
    {"account_id": "1003", "account_name": "shared-services", "business_unit": "shared-ops"},
]

products = ["workflow-intelligence", "ai-assistant", "reporting-api"]
services = [
    "Amazon EC2",
    "AWS Fargate",
    "Amazon RDS",
    "Amazon S3",
    "Amazon CloudWatch",
    "AWS Lambda",
    "AWS Glue",
    "Amazon Athena",
    "Data Transfer",
    "NAT Gateway",
    "Amazon EBS",
    "Support / Shared Platform",
]

teams = ["platform", "data", "product-analytics", "ml-engineering"]
regions = ["us-east-1"]
environments = ["dev", "staging", "prod"]

product_customer_base = {
    "workflow-intelligence": 1200,
    "ai-assistant": 800,
    "reporting-api": 500,
}

product_workflow_multiplier = {
    "workflow-intelligence": 18,
    "ai-assistant": 30,
    "reporting-api": 10,
}

service_cost_multiplier = {
    "Amazon EC2": 0.00022,
    "AWS Fargate": 0.00025,
    "Amazon RDS": 0.00018,
    "Amazon S3": 0.00005,
    "Amazon CloudWatch": 0.00003,
    "AWS Lambda": 0.00008,
    "AWS Glue": 0.00004,
    "Amazon Athena": 0.00003,
    "Data Transfer": 0.00006,
    "NAT Gateway": 0.00007,
    "Amazon EBS": 0.00005,
    "Support / Shared Platform": 0.00002,
}

usage_unit_map = {
    "Amazon EC2": "compute-hours",
    "AWS Fargate": "compute-hours",
    "Amazon RDS": "db-hours",
    "Amazon S3": "gb-month-proxy",
    "Amazon CloudWatch": "logs-metrics-events",
    "AWS Lambda": "requests",
    "AWS Glue": "job-runs",
    "Amazon Athena": "queries",
    "Data Transfer": "gb-transfer",
    "NAT Gateway": "gb-processed",
    "Amazon EBS": "gb-month-proxy",
    "Support / Shared Platform": "shared-unit",
}

env_factor = {"dev": 0.25, "staging": 0.45, "prod": 1.0}
env_sampling_weight = {"dev": 0.28, "staging": 0.32, "prod": 0.40}


def acquisition_phase_for_date(d):
    if d < pd.Timestamp("2025-11-15"):
        return "pre-integration"
    if d < pd.Timestamp("2026-01-01"):
        return "integration"
    return "post-integration"


def seasonal_factor(d):
    day_idx = (d - dates[0]).days
    return 1 + 0.04 * np.sin(day_idx / 14)


def growth_factor(d):
    return 1 + ((d - dates[0]).days / len(dates)) * 0.18


def maybe_missing(value, prob):
    return value if np.random.rand() > prob else None


def maybe_inconsistent_env(env):
    if np.random.rand() < 0.06:
        return np.random.choice(["production", "stage", None], p=[0.45, 0.35, 0.20])
    return env


def choose_sla(product_name, enterprise_flag):
    if enterprise_flag == 1:
        return np.random.choice(["premium", "enterprise"], p=[0.35, 0.65])
    if product_name == "reporting-api":
        return np.random.choice(["standard", "premium"], p=[0.75, 0.25])
    return np.random.choice(["standard", "premium"], p=[0.6, 0.4])


def sample_environment():
    return np.random.choice(
        list(env_sampling_weight.keys()),
        p=list(env_sampling_weight.values())
    )

In [25]:
rows = []

for d in dates:
    for account in accounts:
        for product in products:
            for service in services:
                # Sampling: keep some rows, drop others to land near TARGET_ROWS
                if np.random.rand() > 0.40:
                    continue

                env = sample_environment()
                enterprise_customer_flag = np.random.choice([0, 1], p=[0.7, 0.3])
                team = np.random.choice(teams)
                region = regions[0]

                # -------------------------------
                # BUSINESS SCALE: customers, workflows, revenue
                # -------------------------------
                base_customers = product_customer_base[product] * growth_factor(d) * BUSINESS_SCALE
                customers = max(
                    40,
                    int(np.random.normal(base_customers, 120))
                )

                base_workflows = (
                    customers
                    * product_workflow_multiplier[product]
                    * env_factor[env]
                    * seasonal_factor(d)
                )
                workflows_processed = max(
                    200,
                    int(base_workflows * np.random.uniform(0.9, 1.3))
                )

                # Usage quantity tied to workflows, with some per-service adjustments
                usage_quantity = workflows_processed * np.random.uniform(0.8, 1.2)

                if service == "Amazon S3":
                    usage_quantity *= 0.55 * (1 + ((d - dates[0]).days / len(dates)) * 0.25)
                elif service == "Amazon RDS":
                    usage_quantity *= 0.9
                elif service in ["Data Transfer", "NAT Gateway"]:
                    usage_quantity *= 0.65
                elif service == "Amazon CloudWatch":
                    usage_quantity *= 0.75
                elif service in ["AWS Glue", "Amazon Athena"]:
                    usage_quantity *= 0.18
                elif service == "Support / Shared Platform":
                    usage_quantity *= 0.12

                # -------------------------------
                # COST SCALE: base cost + anomalies + global cost multiplier
                # -------------------------------
                base_cost = (
                    usage_quantity
                    * service_cost_multiplier[service]
                    * np.random.uniform(0.92, 1.16)
                )

                # Planted anomalies (pre-inefficiency)
                if service == "Amazon RDS" and env in ["dev", "staging"] and np.random.rand() < 0.08:
                    base_cost *= 3.5

                if service == "Amazon CloudWatch" and pd.Timestamp("2025-12-08") <= d <= pd.Timestamp("2025-12-16"):
                    base_cost *= 2.8

                if (
                    service in ["Data Transfer", "NAT Gateway"]
                    and account["account_name"] == "acquired-ai-platform"
                    and d >= pd.Timestamp("2025-11-20")
                ):
                    base_cost *= 1.9

                if account["account_name"] == "shared-services" and service == "Support / Shared Platform":
                    base_cost *= 2.2

                # Extra inefficiency multiplier for post-acquisition behavior
                phase = acquisition_phase_for_date(d)
                inefficiency_factor = 1.0
                if phase in ["integration", "post-integration"]:
                    inefficiency_factor *= INEFFICIENCY_MULTIPLIER

                cost_usd = base_cost * COST_SCALE * inefficiency_factor

                # -------------------------------
                # Revenue, budget, forecast (scaled)
                # -------------------------------
                monthly_revenue_proxy = customers * np.random.uniform(70, 180) * BUSINESS_SCALE
                budget_amount = monthly_revenue_proxy * np.random.uniform(0.09, 0.16)
                forecast_baseline_cost = budget_amount * np.random.uniform(0.92, 1.08)

                # Tagging imperfections
                tag_owner = maybe_missing(team, 0.10)
                tag_environment = maybe_inconsistent_env(env)
                tag_product = maybe_missing(product, 0.08)
                tag_cost_center = maybe_missing(account["business_unit"], 0.07)

                allocation_status = "allocated"
                if account["account_name"] == "shared-services" or any(
                    v is None for v in [tag_owner, tag_product, tag_cost_center]
                ):
                    allocation_status = np.random.choice(
                        ["allocated", "partially-allocated", "unallocated"],
                        p=[0.35, 0.40, 0.25],
                    )

                anomaly_flag = 0
                if (
                    (service == "Amazon CloudWatch" and pd.Timestamp("2025-12-08") <= d <= pd.Timestamp("2025-12-16"))
                    or (
                        service in ["Data Transfer", "NAT Gateway"]
                        and account["account_name"] == "acquired-ai-platform"
                        and d >= pd.Timestamp("2025-11-20")
                    )
                    or (service == "Amazon RDS" and env in ["dev", "staging"] and cost_usd > 200)
                ):
                    anomaly_flag = 1

                rows.append(
                    {
                        "usage_date": d.date(),
                        "billing_period": d.strftime("%Y-%m"),
                        "account_id": account["account_id"],
                        "account_name": account["account_name"],
                        "business_unit": account["business_unit"],
                        "environment": env,
                        "region": region,
                        "service_name": service,
                        "product_name": product,
                        "team": team,
                        "cost_usd": round(cost_usd, 2),
                        "usage_quantity": round(usage_quantity, 2),
                        "usage_unit": usage_unit_map[service],
                        "monthly_revenue_proxy": round(monthly_revenue_proxy, 2),
                        "active_customers": customers,
                        "workflows_processed": workflows_processed,
                        "enterprise_customer_flag": enterprise_customer_flag,
                        "sla_tier": choose_sla(product, enterprise_customer_flag),
                        "budget_amount": round(budget_amount, 2),
                        "forecast_baseline_cost": round(forecast_baseline_cost, 2),
                        "acquisition_phase": phase,
                        "tag_owner": tag_owner,
                        "tag_environment": tag_environment,
                        "tag_product": tag_product,
                        "tag_cost_center": tag_cost_center,
                        "allocation_status": allocation_status,
                        "anomaly_flag": anomaly_flag,
                    }
                )

df = pd.DataFrame(rows)

if len(df) < TARGET_ROWS:
    raise ValueError(f"Generated only {len(df)} rows; increase combinations or sampling rate. Got {len(df)} rows.")

df = (
    df.sample(n=TARGET_ROWS, random_state=SEED)
      .sort_values(["usage_date", "account_id", "service_name"])
      .reset_index(drop=True)
)

df.shape, df.head()

((5000, 27),
    usage_date billing_period account_id      account_name  business_unit  \
 0  2025-10-01        2025-10       1001  legacy-core-prod  core-platform   
 1  2025-10-01        2025-10       1001  legacy-core-prod  core-platform   
 2  2025-10-01        2025-10       1001  legacy-core-prod  core-platform   
 3  2025-10-01        2025-10       1001  legacy-core-prod  core-platform   
 4  2025-10-01        2025-10       1001  legacy-core-prod  core-platform   
 
   environment     region       service_name           product_name  \
 0     staging  us-east-1        AWS Fargate           ai-assistant   
 1         dev  us-east-1        AWS Fargate  workflow-intelligence   
 2        prod  us-east-1         AWS Lambda           ai-assistant   
 3         dev  us-east-1         AWS Lambda  workflow-intelligence   
 4     staging  us-east-1  Amazon CloudWatch          reporting-api   
 
                 team  ...    sla_tier  budget_amount forecast_baseline_cost  \
 0     ml-engin

In [26]:
dictionary_rows = [
    ("usage_date", "Date of usage at daily granularity"),
    ("billing_period", "Billing month label in YYYY-MM format"),
    ("account_id", "AWS account identifier"),
    ("account_name", "Human-readable AWS account label"),
    ("business_unit", "Business unit or portfolio grouping"),
    ("environment", "Deployment environment"),
    ("region", "AWS region"),
    ("service_name", "AWS service or cost category"),
    ("product_name", "Product or workload slice"),
    ("team", "Owning or associated team"),
    ("cost_usd", "Daily cost in USD (scaled for post-acquisition scenario)"),
    ("usage_quantity", "Usage quantity proxy associated with the service"),
    ("usage_unit", "Unit for usage_quantity"),
    ("monthly_revenue_proxy", "Revenue proxy associated with the workload (scaled with business size)"),
    ("active_customers", "Estimated active customer count (scaled with business size)"),
    ("workflows_processed", "Estimated workflows processed (scaled with business size)"),
    ("enterprise_customer_flag", "Whether enterprise customer mix is present"),
    ("sla_tier", "Service level expectation"),
    ("budget_amount", "Budgeted cloud cost amount for the relevant scope"),
    ("forecast_baseline_cost", "Forecast baseline for the relevant scope"),
    ("acquisition_phase", "Pre-integration, integration, or post-integration"),
    ("tag_owner", "Synthetic owner tag"),
    ("tag_environment", "Synthetic environment tag (includes some inconsistent values)"),
    ("tag_product", "Synthetic product tag"),
    ("tag_cost_center", "Synthetic cost center tag"),
    ("allocation_status", "Allocated, partially-allocated, or unallocated"),
    ("anomaly_flag", "Indicates whether the row is part of a planted anomaly pattern"),
]

dictionary_df = pd.DataFrame(dictionary_rows, columns=["column_name", "description"])

df.to_csv(BILLING_PATH, index=False)
dictionary_df.to_csv(DICTIONARY_PATH, index=False)

print(f"Wrote {len(df)} rows to {BILLING_PATH}")
print(f"Wrote data dictionary to {DICTIONARY_PATH}")
df.head(10)

Wrote 5000 rows to billing.csv
Wrote data dictionary to data_dictionary.csv


,usage_date,billing_period,account_id,account_name,business_unit,environment,region,service_name,product_name,team,...,sla_tier,budget_amount,forecast_baseline_cost,acquisition_phase,tag_owner,tag_environment,tag_product,tag_cost_center,allocation_status,anomaly_flag
0,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,staging,us-east-1,AWS Fargate,ai-assistant,ml-engineering,...,premium,98400.73,97716.41,pre-integration,ml-engineering,staging,ai-assistant,core-platform,allocated,0
1,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,dev,us-east-1,AWS Fargate,workflow-intelligence,product-analytics,...,standard,271679.01,258624.24,pre-integration,product-analytics,dev,None,core-platform,allocated,0
2,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,prod,us-east-1,AWS Lambda,ai-assistant,data,...,enterprise,216109.16,199963.24,pre-integration,data,prod,ai-assistant,core-platform,allocated,0
3,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,dev,us-east-1,AWS Lambda,workflow-intelligence,product-analytics,...,standard,194208.81,191883.15,pre-integration,product-analytics,dev,None,core-platform,partially-allocated,0
4,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,staging,us-east-1,Amazon CloudWatch,reporting-api,ml-engineering,...,premium,101965.35,102659.37,pre-integration,ml-engineering,staging,reporting-api,core-platform,allocated,0
5,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,prod,us-east-1,Amazon EBS,ai-assistant,platform,...,standard,118689.82,122407.83,pre-integration,platform,prod,ai-assistant,core-platform,allocated,0
6,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,dev,us-east-1,Amazon EBS,workflow-intelligence,platform,...,premium,384937.40,381824.07,pre-integration,None,dev,workflow-intelligence,core-platform,partially-allocated,0
7,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,prod,us-east-1,Amazon EC2,workflow-intelligence,platform,...,enterprise,164954.12,153246.65,pre-integration,platform,prod,None,core-platform,partially-allocated,0
8,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,prod,us-east-1,Amazon EC2,ai-assistant,platform,...,enterprise,154185.56,142477.79,pre-integration,platform,stage,ai-assistant,core-platform,allocated,0
9,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,prod,us-east-1,Amazon RDS,ai-assistant,product-analytics,...,standard,89451.03,88269.00,pre-integration,product-analytics,prod,ai-assistant,core-platform,allocated,0


In [27]:
df

,usage_date,billing_period,account_id,account_name,business_unit,environment,region,service_name,product_name,team,...,sla_tier,budget_amount,forecast_baseline_cost,acquisition_phase,tag_owner,tag_environment,tag_product,tag_cost_center,allocation_status,anomaly_flag
0,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,staging,us-east-1,AWS Fargate,ai-assistant,ml-engineering,...,premium,98400.73,97716.41,pre-integration,ml-engineering,staging,ai-assistant,core-platform,allocated,0
1,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,dev,us-east-1,AWS Fargate,workflow-intelligence,product-analytics,...,standard,271679.01,258624.24,pre-integration,product-analytics,dev,None,core-platform,allocated,0
2,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,prod,us-east-1,AWS Lambda,ai-assistant,data,...,enterprise,216109.16,199963.24,pre-integration,data,prod,ai-assistant,core-platform,allocated,0
3,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,dev,us-east-1,AWS Lambda,workflow-intelligence,product-analytics,...,standard,194208.81,191883.15,pre-integration,product-analytics,dev,None,core-platform,partially-allocated,0
4,2025-10-01,2025-10,1001,legacy-core-prod,core-platform,staging,us-east-1,Amazon CloudWatch,reporting-api,ml-engineering,...,premium,101965.35,102659.37,pre-integration,ml-engineering,staging,reporting-api,core-platform,allocated,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2026-01-28,2026-01,1003,shared-services,shared-ops,prod,us-east-1,Amazon RDS,reporting-api,product-analytics,...,standard,136186.09,142806.35,post-integration,product-analytics,prod,reporting-api,shared-ops,partially-allocated,0
4996,2026-01-28,2026-01,1003,shared-services,shared-ops,dev,us-east-1,Amazon RDS,workflow-intelligence,data,...,enterprise,205486.34,210626.57,post-integration,data,dev,workflow-intelligence,shared-ops,allocated,0
4997,2026-01-28,2026-01,1003,shared-services,shared-ops,prod,us-east-1,Amazon RDS,ai-assistant,product-analytics,...,standard,129461.87,135340.90,post-integration,product-analytics,prod,ai-assistant,shared-ops,partially-allocated,0
4998,2026-01-28,2026-01,1003,shared-services,shared-ops,prod,us-east-1,NAT Gateway,ai-assistant,platform,...,standard,115124.18,123360.32,post-integration,None,prod,ai-assistant,shared-ops,allocated,0


In [28]:
df.groupby("service_name")["cost_usd"].sum().sort_values(ascending=False).head(10)

service_name
AWS Fargate          28301.39
Amazon EC2           27072.91
Amazon RDS           20382.69
AWS Lambda            8907.14
NAT Gateway           6218.00
Amazon EBS            5745.86
Data Transfer         5382.88
Amazon S3             3448.15
Amazon CloudWatch     2754.06
AWS Glue               821.36
Name: cost_usd, dtype: float64

In [29]:
df.groupby("product_name")["cost_usd"].sum().sort_values(ascending=False).head(10)

product_name
ai-assistant             51472.71
workflow-intelligence    47452.29
reporting-api            11095.40
Name: cost_usd, dtype: float64

In [30]:
df.groupby("business_unit")["cost_usd"].sum().sort_values(ascending=False).head(10)

business_unit
ai-products      39302.82
core-platform    35971.40
shared-ops       34746.18
Name: cost_usd, dtype: float64

In [31]:
df.groupby("account_name")["cost_usd"].sum().sort_values(ascending=False).head(10)

account_name
acquired-ai-platform    39302.82
legacy-core-prod        35971.40
shared-services         34746.18
Name: cost_usd, dtype: float64

In [32]:
df.groupby("team")["cost_usd"].sum().sort_values(ascending=False).head(10)

team
product-analytics    30627.82
ml-engineering       26987.99
platform             26511.97
data                 25892.62
Name: cost_usd, dtype: float64

In [33]:
df.groupby("environment")["cost_usd"].sum().sort_values(ascending=False).head(10)

environment
prod       69316.63
staging    27453.06
dev        13250.71
Name: cost_usd, dtype: float64

In [34]:
df.groupby("sla_tier")["cost_usd"].sum().sort_values(ascending=False).head(10)

sla_tier
standard      45765.23
premium       42862.72
enterprise    21392.45
Name: cost_usd, dtype: float64

In [35]:
df.groupby("acquisition_phase")["cost_usd"].sum().sort_values(ascending=False).head(10)

acquisition_phase
integration         43773.66
post-integration    35007.89
pre-integration     31238.85
Name: cost_usd, dtype: float64

In [36]:
# 4. Non-prod RDS
df[df["service_name"] == "Amazon RDS"].groupby("environment")["cost_usd"].sum().sort_values(ascending=False)

environment
prod       11144.14
staging     6311.26
dev         2927.29
Name: cost_usd, dtype: float64

In [37]:
# 5. Transfer + NAT by account
df[df["service_name"].isin(["Data Transfer", "NAT Gateway"])].groupby("account_name")["cost_usd"].sum().sort_values(ascending=False)

account_name
acquired-ai-platform    5287.54
shared-services         3213.67
legacy-core-prod        3099.67
Name: cost_usd, dtype: float64

In [38]:
# 6. CloudWatch over time
df[df["service_name"] == "Amazon CloudWatch"].groupby("usage_date")["cost_usd"].sum().sort_values(ascending=False).head(15)

usage_date
2025-12-12    97.96
2025-12-13    93.93
2026-01-21    63.32
2025-11-20    58.30
2025-12-14    55.79
2025-12-16    55.51
2026-01-24    55.01
2025-11-24    53.16
2025-12-11    47.43
2025-12-04    47.14
2026-01-10    46.99
2025-12-28    46.73
2026-01-12    46.53
2026-01-25    44.17
2025-12-29    41.18
Name: cost_usd, dtype: float64

In [39]:
# 7. Allocation quality
df["allocation_status"].value_counts(dropna=False)

allocation_status
allocated              3484
partially-allocated     940
unallocated             576
Name: count, dtype: int64

In [40]:
df["cost_usd"].sum()

np.float64(110020.4)